# IND320 – Project work, part 3: Data quality
**Course:** IND320 Data to decision (NMBU, autumn 2026)
**Student:** Zarar Zafar

## Links
- **GitHub repository:** [https://github.com/zararzafar/zarar](https://github.com/zararzafar/zarar) (work for this part is on the branch `part3`)
- **Streamlit app (part 3 version, branch `part3`):** [https://zarar-ind320-part3.streamlit.app](https://zarar-ind320-part3.streamlit.app)
- **Streamlit app (version from part 1, branch `main`):** [https://zarar-ind320.streamlit.app](https://zarar-ind320.streamlit.app)

## Contents
1. Work log
2. Setup
3. Shapefiles for price areas and precipitation areas
4. Random locations in each area
5. Weather data from open-meteo (ERA5)
6. Outliers in precipitation (DCT + SPC)
7. Anomalies in temperature (Local Outlier Factor)
8. STL decomposition of the reservoir fill rate
9. Spectrogram of the reservoir fill rate

## 1. Work log
*(300–500 words, describing both the Jupyter Notebook and the Streamlit work)*

Part 3 is about data quality: finding outliers and anomalies in weather data and describing the
structure of the reservoir data. As before, the work is kept on a separate GitHub branch (`part3`)
until the peer review and feedback for the previous part are finished.

The first task was to find geographical areas that match the Magasinstatistikk data. In the NVE map
services behind atlas.nve.no, the layer *Budomraade* contains the five electricity price areas NO1–NO5
and the layer *NVE_Region* contains NVE's regions RØ, RS, RV, RM and RN. Following the assignment, the
precipitation areas were built by joining RS and RØ into the eastern area (VASS1), using RV as the
western area (VASS2) and joining RM and RN into the northern area (VASS3). The layers were downloaded
as GeoJSON through the ArcGIS REST interface and saved as shapefiles. 1000 random locations were then
sampled inside each area. The sampling is done in a metric projection (UTM 33N), so the points are
evenly spread over the area and not squeezed towards the north as they would be in longitude/latitude.

The weather data come from the open-meteo archive API with the ERA5 reanalysis model. The download
function takes a region and a time interval, looks up the sampled locations and downloads hourly
temperature, precipitation, rain, snowfall, wind speed and relative humidity. To stay within the free
API limits, a random subset of the 1000 locations is used (10 by default), and all locations are sent
in one request. The function was applied to NO3 for 2024 and the values were averaged over locations.

For precipitation, a high-pass filter with the Discrete Cosine Transform removes the slow seasonal
variation. Robust statistics (median and scaled median absolute deviation) of these seasonally adjusted
variations give Statistical Process Control limits, which are drawn around the slowly varying part of
the series, and points outside them are marked as outliers. Precipitation is very skewed, so most
outliers are heavy-rain hours above the upper limit. For temperature, the Local Outlier Factor with 1 %
contamination finds hours that differ from their neighbours in time, typically sudden warm or cold
spells. For the fill rate of NO3, STL separates trend, yearly season and residual, and the spectrogram
confirms a strong component at one cycle per year. All methods are wrapped in functions with
parameters and sensible defaults, and each function is tested with more than one setting.

In the Streamlit app, the analysis functions are placed in a shared module so that the same code can
be used on the new pages: one page with tabs for STL and the spectrogram, and one page with tabs for
the SPC and LOF analyses, with the weather download following the area selector used for the
reservoir data.

## 2. Setup
All packages are available in Google Colab. Locally: `pip install requests pandas numpy matplotlib geopandas scipy scikit-learn statsmodels`.

In [ ]:
# Libraries used in this notebook
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import geopandas as gpd

from scipy.fft import dct, idct                       # Discrete Cosine Transform
from scipy.signal import spectrogram                  # spectrogram (short-time Fourier analysis)
from scipy.stats import median_abs_deviation          # robust spread
from sklearn.neighbors import LocalOutlierFactor      # anomaly detection
from statsmodels.tsa.seasonal import STL              # Seasonal-Trend decomposition using LOESS

RNG_SEED = 320                                        # fixed seed so the random sampling is reproducible

## 3. Shapefiles for price areas and precipitation areas
The map layers shown in [atlas.nve.no](https://atlas.nve.no) are served by NVE's ArcGIS servers:
- **Price areas** (NO1–NO5): layer *Budomraade* in the service `Nettomraader`
- **NVE regions** (RØ, RS, RV, RM, RN): layer *NVE_Region* in the service `Administrasjon2`

The layers are downloaded as GeoJSON in longitude/latitude (EPSG:4326) and saved as shapefiles.

In [ ]:
# ArcGIS REST layers behind atlas.nve.no
PRICE_AREA_URL = "https://kart.nve.no/enterprise/rest/services/Nettomraader/MapServer/0"
NVE_REGION_URL = "https://gis3.nve.no/map/rest/services/Mapservices/Administrasjon2/MapServer/0"

def download_layer(url):
    """Download all features of an ArcGIS map layer as a GeoDataFrame in EPSG:4326."""
    params = {"where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson",
              "maxAllowableOffset": 0.002, "geometryPrecision": 4}   # light simplification (~200 m)
    response = requests.get(url + "/query", params=params, timeout=120)
    response.raise_for_status()
    return gpd.read_file(io.StringIO(response.text))

# Price areas: 'NO 1' -> 'NO1'
price_areas = download_layer(PRICE_AREA_URL)
price_areas["region"] = price_areas["budomr"].str.replace(" ", "")
price_areas = price_areas[["region", "geometry"]].sort_values("region")

# Precipitation areas built from NVE's regions, as described in the assignment:
# East = RS + RØ (VASS1), West = RV (VASS2), North = RM + RN (VASS3)
nve_regions = download_layer(NVE_REGION_URL)
to_precip_area = {"RS": "VASS1", "RØ": "VASS1", "RV": "VASS2", "RM": "VASS3", "RN": "VASS3"}
nve_regions["region"] = nve_regions["REGION"].map(to_precip_area)
precip_areas = nve_regions[["region", "geometry"]].dissolve(by="region").reset_index()

# Save as shapefiles (one folder per layer)
price_areas.to_file("price_areas.shp")
precip_areas.to_file("precipitation_areas.shp")

areas = pd.concat([price_areas, precip_areas], ignore_index=True)
print(areas["region"].tolist())

In [ ]:
# Show both sets of areas
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, gdf, title in [(axes[0], price_areas, "Price areas (NO1–NO5)"),
                       (axes[1], precip_areas, "Precipitation areas (VASS1–VASS3)")]:
    gdf.plot(column="region", ax=ax, legend=True, cmap="tab10", edgecolor="white", linewidth=0.3)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.tight_layout()
plt.show()

## 4. Random locations in each area
1000 random points are drawn inside each area. Sampling is done in UTM zone 33N (EPSG:25833), an
equal-area-like metric projection, so the points are evenly spread over the area. The points are then
converted back to longitude/latitude for the weather API.

In [ ]:
def sample_locations(polygon_utm, n=1000, rng=None):
    """Draw n random points inside a polygon given in a metric projection (rejection sampling)."""
    rng = rng or np.random.default_rng(RNG_SEED)
    minx, miny, maxx, maxy = polygon_utm.bounds
    points = []
    while len(points) < n:
        x = rng.uniform(minx, maxx, 4 * n)
        y = rng.uniform(miny, maxy, 4 * n)
        candidates = gpd.GeoSeries(gpd.points_from_xy(x, y), crs=25833)
        points.extend(candidates[candidates.within(polygon_utm)])   # keep points inside the area
    return gpd.GeoSeries(points[:n], crs=25833).to_crs(4326)

rng = np.random.default_rng(RNG_SEED)
areas_utm = areas.to_crs(25833)
LOCATIONS = {}            # region -> DataFrame with 1000 longitude/latitude pairs
for region, polygon in zip(areas_utm["region"], areas_utm.geometry):
    pts = sample_locations(polygon, 1000, rng)
    LOCATIONS[region] = pd.DataFrame({"longitude": pts.x.round(4), "latitude": pts.y.round(4)})

print({region: len(df) for region, df in LOCATIONS.items()})
LOCATIONS["NO3"].head()

In [ ]:
# Check: the sampled locations in NO3 and VASS3
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, region, gdf in [(axes[0], "NO3", price_areas), (axes[1], "VASS3", precip_areas)]:
    gdf[gdf["region"] == region].plot(ax=ax, color="lightgrey", edgecolor="grey")
    ax.scatter(LOCATIONS[region]["longitude"], LOCATIONS[region]["latitude"], s=2, color="tab:red")
    ax.set_title(f"1000 random locations in {region}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.tight_layout()
plt.show()

## 5. Weather data from open-meteo (ERA5)
The [open-meteo archive API](https://open-meteo.com/en/docs/historical-weather-api) provides hourly
historical reanalysis data. The model is chosen with `models=era5`. Several locations can be requested
in one call by giving comma-separated latitudes and longitudes.

To respect open-meteo's free usage limits, the function uses a random subset of the 1000 locations
(`n_locations`, default 10). With `n_locations=1` it downloads a single location.

In [ ]:
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_VARIABLES = ["temperature_2m", "precipitation", "rain", "snowfall",
                    "wind_speed_10m", "relative_humidity_2m"]

def download_weather(region, start_date, end_date, n_locations=10, seed=RNG_SEED):
    """Download hourly ERA5 weather for random locations in a region.

    region     : region identifier, e.g. 'NO3' or 'VASS1' (key in LOCATIONS)
    start_date : first day, 'YYYY-MM-DD'
    end_date   : last day, 'YYYY-MM-DD'
    Returns a long DataFrame with one row per location and hour.
    """
    points = LOCATIONS[region].sample(n_locations, random_state=seed)
    params = {
        "latitude": ",".join(points["latitude"].astype(str)),
        "longitude": ",".join(points["longitude"].astype(str)),
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARIABLES),
        "models": "era5",
        "timezone": "UTC",
    }
    response = requests.get(ARCHIVE_URL, params=params, timeout=300)
    response.raise_for_status()
    result = response.json()
    if isinstance(result, dict):            # one location gives a dict, several give a list
        result = [result]

    frames = []
    for i, location in enumerate(result):
        df = pd.DataFrame(location["hourly"])
        df["time"] = pd.to_datetime(df["time"])
        df["location"] = i
        df["latitude"] = location["latitude"]
        df["longitude"] = location["longitude"]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

# Apply the function to NO3 for 2024
weather = download_weather("NO3", "2024-01-01", "2024-12-31")
print(weather.shape, "-", weather["location"].nunique(), "locations")
weather.head()

In [ ]:
# Average over the locations: one hourly series per variable for NO3
weather_avg = weather.groupby("time")[HOURLY_VARIABLES].mean()
weather_avg.describe().round(2)

## 6. Outliers in precipitation (DCT + SPC)
First the average precipitation as a function of time:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weather_avg.index, weather_avg["precipitation"], linewidth=0.6)
ax.set_title("Average hourly precipitation in NO3, 2024 (ERA5)")
ax.set_xlabel("Time")
ax.set_ylabel("Precipitation (mm per hour)")
ax.grid(alpha=0.3)
plt.show()

The DCT of the series is computed and the lowest `cutoff` coefficients are set to zero (high-pass
filter). The inverse DCT gives the seasonally adjusted variations (SATV). SPC limits are
median ± `n_std` × robust standard deviation (MAD scaled to the normal distribution) of the SATV for
the whole year. The limits are drawn around the slowly varying part of the series; the SATV values
themselves are not plotted.

In [ ]:
def precipitation_spc(precip: pd.Series, cutoff: int = 50, n_std: float = 3.0):
    """Statistical Process Control (SPC) for precipitation, based on DCT high-pass filtering.

    precip : hourly (average) precipitation with a DatetimeIndex
    cutoff : number of the lowest DCT coefficients that are removed (the high-pass cut-off).
             Index k corresponds to a period of about 2*N/k samples, so for one year of hourly
             data the default 50 removes variations slower than roughly two weeks.
    n_std  : number of robust standard deviations between the centre and the SPC limits

    Returns (fig, summary) where summary is a dict with statistics and a table of the outliers.
    """
    values = precip.to_numpy(dtype=float)

    # DCT of the whole series; setting the lowest coefficients to zero gives a high-pass filter.
    coeffs = dct(values, norm="ortho")
    coeffs[:cutoff] = 0
    satv = idct(coeffs, norm="ortho")          # seasonally adjusted variations (SATV)
    slow = values - satv                       # the slowly varying (seasonal) part

    # Robust statistics of the SATV for the whole period: median and MAD scaled to a standard deviation
    centre = np.median(satv)
    robust_sd = median_abs_deviation(satv, scale="normal")
    upper_satv = centre + n_std * robust_sd
    lower_satv = centre - n_std * robust_sd
    is_outlier = (satv > upper_satv) | (satv < lower_satv)

    # The SPC limits are drawn around the slowly varying part of the original series,
    # so the SATV values themselves are never plotted.
    upper = slow + upper_satv
    lower = slow + lower_satv

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(precip.index, values, color="tab:blue", linewidth=0.6, label="Precipitation")
    ax.plot(precip.index, upper, color="tab:green", linewidth=1, label=f"SPC limits (±{n_std:g} robust SD)")
    ax.plot(precip.index, lower, color="tab:green", linewidth=1)
    ax.scatter(precip.index[is_outlier], values[is_outlier], color="tab:red", s=12, zorder=3,
               label=f"Outliers ({is_outlier.sum()})")
    ax.set_title(f"Precipitation with SPC limits (DCT cut-off = {cutoff})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Precipitation (mm per hour)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()

    outliers = pd.DataFrame({"precipitation_mm": values[is_outlier],
                             "satv": satv[is_outlier]}, index=precip.index[is_outlier])
    summary = {
        "n_values": len(values),
        "n_outliers": int(is_outlier.sum()),
        "share_outliers": float(is_outlier.mean()),
        "n_above": int((satv > upper_satv).sum()),
        "n_below": int((satv < lower_satv).sum()),
        "satv_median": float(centre),
        "satv_robust_sd": float(robust_sd),
        "cutoff": cutoff,
        "n_std": n_std,
        "outliers": outliers,
    }
    return fig, summary

In [ ]:
# Test 1: default parameters
fig, spc = precipitation_spc(weather_avg["precipitation"])
plt.show()
print({k: v for k, v in spc.items() if k != "outliers"})
spc["outliers"].sort_values("precipitation_mm", ascending=False).head(10)

In [ ]:
# Test 2: lower cut-off (keeps more of the slow variation in the SATV) and wider limits
fig, spc2 = precipitation_spc(weather_avg["precipitation"], cutoff=20, n_std=4)
plt.show()
print("Outliers with cutoff=20, n_std=4:", spc2["n_outliers"], f"({spc2['share_outliers']:.1%})")

## 7. Anomalies in temperature (Local Outlier Factor)
First the average temperature as a function of time:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weather_avg.index, weather_avg["temperature_2m"], linewidth=0.6)
ax.set_title("Average hourly temperature in NO3, 2024 (ERA5)")
ax.set_xlabel("Time")
ax.set_ylabel("Temperature 2 m (°C)")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def temperature_lof(temp: pd.Series, contamination: float = 0.01, n_neighbors: int = 48,
                    time_scale_days: float = 3.0):
    """Anomaly detection for temperature with the Local Outlier Factor (LOF).

    Each observation is described by two standardised features: its time and its temperature.
    Time is scaled so that `time_scale_days` days count as much as one standard deviation of the
    temperature, which makes the method compare each value with its neighbours in time.

    contamination : proportion of points that are classified as anomalies (default 1 %)
    n_neighbors   : number of neighbours used by LOF (48 = two days of hourly values)

    Returns (fig, summary).
    """
    values = temp.to_numpy(dtype=float)
    hours = (temp.index - temp.index[0]) / pd.Timedelta(hours=1)
    X = np.column_stack([
        np.asarray(hours, dtype=float) / (24 * time_scale_days),
        (values - values.mean()) / values.std(),
    ])

    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    labels = lof.fit_predict(X)                 # -1 = anomaly, 1 = normal
    scores = -lof.negative_outlier_factor_      # larger score = more anomalous
    is_anomaly = labels == -1

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(temp.index, values, color="tab:blue", linewidth=0.6, label="Temperature")
    ax.scatter(temp.index[is_anomaly], values[is_anomaly], color="tab:red", s=12, zorder=3,
               label=f"LOF anomalies ({is_anomaly.sum()})")
    ax.set_title(f"Temperature with Local Outlier Factor anomalies (proportion = {contamination:.1%})")
    ax.set_xlabel("Time")
    ax.set_ylabel("Temperature 2 m (°C)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()

    anomalies = pd.DataFrame({"temperature_C": values[is_anomaly], "lof_score": scores[is_anomaly]},
                             index=temp.index[is_anomaly]).sort_values("lof_score", ascending=False)
    summary = {
        "n_values": len(values),
        "n_anomalies": int(is_anomaly.sum()),
        "share_anomalies": float(is_anomaly.mean()),
        "contamination": contamination,
        "n_neighbors": n_neighbors,
        "mean_temp_anomalies": float(values[is_anomaly].mean()) if is_anomaly.any() else float("nan"),
        "mean_temp_all": float(values.mean()),
        "anomalies": anomalies,
    }
    return fig, summary

In [ ]:
# Test 1: default proportion of outliers (1 %)
fig, lof = temperature_lof(weather_avg["temperature_2m"])
plt.show()
print({k: v for k, v in lof.items() if k != "anomalies"})
lof["anomalies"].head(10)

In [ ]:
# Test 2: a smaller proportion of outliers (0.5 %)
fig, lof2 = temperature_lof(weather_avg["temperature_2m"], contamination=0.005)
plt.show()
print("Anomalies with contamination=0.005:", lof2["n_anomalies"])

## 8. STL decomposition of the reservoir fill rate
The weekly fill rate (*fyllingsgrad*) for NO3 is downloaded from the NVE Magasinstatistikk API.

In [ ]:
# Weekly reservoir fill rate for price area NO3 from the NVE API
NVE_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"
reservoirs = pd.DataFrame(requests.get(NVE_URL, timeout=60).json())
reservoirs["dato_Id"] = pd.to_datetime(reservoirs["dato_Id"])

fill_no3 = (reservoirs[(reservoirs["omrType"] == "EL") & (reservoirs["omrnr"] == 3)]
            .sort_values("dato_Id")                       # the rows are not sorted in the API
            .set_index("dato_Id")["fyllingsgrad"]
            .rename("fill_rate"))
print(len(fill_no3), "weeks from", fill_no3.index.min().date(), "to", fill_no3.index.max().date())

In [ ]:
def stl_decomposition(series: pd.Series, period: int = 52, seasonal: int = 13,
                      trend: int = 105, robust: bool = True, title: str = ""):
    """Seasonal-Trend decomposition using LOESS (STL) of a weekly series.

    period   : length of one season (52 weeks = one year)
    seasonal : length of the seasonal smoother (odd, >= 7)
    trend    : length of the trend smoother (odd, > period); 105 weeks is about two years
    robust   : use robust weights that reduce the influence of outliers
    """
    from statsmodels.tsa.seasonal import STL   # imported here so the other functions work without it

    result = STL(series, period=period, seasonal=seasonal, trend=trend, robust=robust).fit()
    fig = result.plot()
    fig.set_size_inches(11, 8)
    fig.suptitle(f"STL decomposition {title} (seasonal={seasonal}, trend={trend}, robust={robust})",
                 y=1.01)
    fig.tight_layout()
    return fig

In [ ]:
# Test 1: default parameters
fig = stl_decomposition(fill_no3, title="of the fill rate in NO3")
plt.show()

In [ ]:
# Test 2: longer seasonal smoother, shorter trend smoother and no robust weights
fig = stl_decomposition(fill_no3, seasonal=31, trend=61, robust=False, title="of the fill rate in NO3")
plt.show()

## 9. Spectrogram of the reservoir fill rate
The spectrogram shows how the frequency content of the fill rate changes over time. With weekly data
the sampling frequency is 52 per year, so the yearly cycle appears at 1 cycle per year.

In [ ]:
def fill_spectrogram(series: pd.Series, window_length: int = 104, overlap: int = 78,
                     samples_per_year: float = 52.0, title: str = ""):
    """Spectrogram of a weekly series.

    window_length : number of weeks in each window (104 = two years)
    overlap       : number of weeks two neighbouring windows have in common
    The frequency axis is in cycles per year, so the yearly cycle is at 1.
    """
    values = series.to_numpy(dtype=float)
    f, t, Sxx = spectrogram(values - values.mean(), fs=samples_per_year,
                            nperseg=window_length, noverlap=overlap)
    # t is in years from the start of the series: convert to dates for the x-axis
    times = series.index[0] + pd.to_timedelta(t * 365.25, unit="D")

    fig, ax = plt.subplots(figsize=(11, 4.5))
    mesh = ax.pcolormesh(times, f, 10 * np.log10(Sxx + 1e-12), shading="gouraud", cmap="viridis")
    fig.colorbar(mesh, ax=ax, label="Power (dB)")
    ax.set_title(f"Spectrogram {title} (window = {window_length} weeks, overlap = {overlap} weeks)")
    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency (cycles per year)")
    fig.tight_layout()
    return fig

In [ ]:
# Test 1: default parameters (two-year windows)
fig = fill_spectrogram(fill_no3, title="of the fill rate in NO3")
plt.show()

In [ ]:
# Test 2: longer windows (four years) with half overlap
fig = fill_spectrogram(fill_no3, window_length=208, overlap=104, title="of the fill rate in NO3")
plt.show()

## Next steps
In part 4 the project will build on these analyses, the weather data and the reservoir data.